In [1]:
from google.colab import drive
import os
import pandas as pd

drive.mount('/content/drive')

base_path = '/content/drive/My Drive/CMPE 255 Data Mining/Dataset'

print(f"Listing contents of '{base_path}':")
if os.path.exists(base_path):
    for item in os.listdir(base_path):
        print(item)
else:
    print(f"Error: The path '{base_path}' does not exist. Please check the folder name and structure in your Google Drive.")

# Read the identified JSON file into a pandas DataFrame
file_name = 'renttherunway_final_data.json'
json_file_path = os.path.join(base_path, file_name)

if os.path.exists(json_file_path):
    try:
        # Added lines=True to handle JSON Lines format
        df = pd.read_json(json_file_path, lines=True)
        print(f"\nSuccessfully loaded '{file_name}' into a pandas DataFrame.")
        print("First 5 rows of the DataFrame:")
        display(df.head())
    except Exception as e:
        print(f"Error reading JSON file '{file_name}': {e}")
else:
    print(f"Error: The file '{json_file_path}' does not exist.")

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
target_feature = 'fit'
n_classes = df[target_feature].unique().shape[0]
df[target_feature].hist(bins=n_classes, edgecolor='black');

In [ ]:
#checking for null values and dropping them.
def check_nulls(data):
    for col in df:
        print(f'Column \'{col}\'. Is null - {data[col].isnull().sum()}')


to_drop = df[df['fit'] == 'fit'].isnull().any(axis=1)
n = to_drop.sum()
to_drop.shape, df.shape
df = df.drop(df[df['fit'] == 'fit'][to_drop].index, axis=0)
print(f'Dropped {n} examples')

check_nulls(df)

In [ ]:
#converting height from feet to inches
def parse_ht(height):
    ht_ = height.split("' ")
    ft_ = float(ht_[0])
    in_ = float(ht_[1].replace("\"",""))
    return (12*ft_) + in_
#converting weight from pounds to kilos.
def pounds_to_kilos(s):
    return int(s.replace('lbs', '')) * 0.45359237

df['height'] = (df['height']
                        .fillna("0' 0\"")
                        .apply(parse_ht))
df['height'][df['height'] == 0] = df['height'].median()

df['weight'] = (df['weight']
                        .fillna('0lbs')
                        .apply(pounds_to_kilos))
df['weight'][df['weight'] == 0.0] = df['weight'].median()

df['user_id'] = pd.to_numeric(df['user_id'])
df['bust size'] = df['bust size'].fillna(df['bust size'].value_counts().index[0])
df['body type'] = df['body type'].fillna(df['body type'].value_counts().index[0])
df['item_id'] = pd.to_numeric(df['item_id'])
df['size'] = pd.to_numeric(df['size'])

df['age'] = pd.to_numeric(df['age'])
df['age'] = df['age'].fillna(df['age'].median())

df['rating'] = pd.to_numeric(df['rating'])
df['rating'] = df['rating'].fillna(df['rating'].median())

df['review_date'] = pd.to_datetime(df['review_date'], format='%B %d, %Y')
#replacing null values of numeric features with median values.
df.info()

In [ ]:
#column mapper
col_mapper = {
    'bust size': 'bust_size',
    'weight': 'usr_weight_kg',
    'rating': 'review_rating',
    'rented for': 'rented_for',
    'body type': 'body_type',
    'category': 'product_category',
    'height': 'usr_height_inchs',
    'size': 'product_size',
    'age': 'usr_age',
}
df.rename(col_mapper, axis=1, inplace=True)

In [ ]:
newdf = df.copy()

In [ ]:
#bust size and category mapper
import re

def parse_bust_size(s):
    m = re.match(r'(\d+)([A-Za-z])(\+?)', s)
    if m:
        return pd.Series(data=[int(m.group(1)), m.group(2).lower()])
    return []

mapper = {
    0: 'bust_size_num',
    1: 'bust_size_cat'
}

temp_df = newdf['bust_size'].apply(parse_bust_size).rename(mapper, axis=1)
temp_df['bust_size_num'] = pd.to_numeric(temp_df['bust_size_num'])
newdf = newdf.join(temp_df)
newdf.drop(['bust_size'], axis=1, inplace=True)




In [ ]:
#bust category mapper
mapper = {
    'a': 1,
    'b': 2,
    'c': 3,
    'd': 4,
    'e': 5,
    'f': 6,
    'g': 7,
    'h': 8,
    'i': 9,
    'j': 10,
}
newdf['bust_size_cat'] = newdf['bust_size_cat'].map(mapper)

newdf.head()


In [ ]:
mapper = {
    'small': -1,
    'fit': 0,
    'large': 1,
}
newdf['fit'] = newdf['fit'].map(mapper)

In [ ]:
#Scaling the data and getting correlation matrix.
from sklearn.preprocessing import scale
import seaborn as sns

numeric_dtypes = {'int64', 'float64'}
numeric_features = [c for c in newdf.columns if str(newdf[c].dtype) in numeric_dtypes]
numeric_features.remove('user_id')
numeric_features.remove('item_id')
numeric_features.remove('review_rating')

cleaned_df_scaled = newdf[numeric_features].copy()
cleaned_df_scaled = pd.DataFrame(scale(cleaned_df_scaled), columns=numeric_features)

corr_matrix = cleaned_df_scaled.corr()
sns.heatmap(corr_matrix)
corr_matrix

In [ ]:
print('Pairs of columns that have correlation greater than 0.5: ')
#finding columns which have correlation value greater than 0.5
lim = 0.5
corr_cols = []
for i in range(corr_matrix.shape[0]):
    for j in range(i + 1, corr_matrix.shape[1]):
        if corr_matrix.iloc[i, j] > lim:
            pair = corr_matrix.columns[i], corr_matrix.columns[j]
            corr_cols.append(pair)
            print('({}, {})'.format(*pair))

print('These columns are to be inspected more closely')
corr_cols



In [ ]:
#plotting box plot between fit and product size.
import matplotlib.pyplot as plt
plt.subplots(figsize=(15,7))
ax=sns.boxplot(x='fit',y='product_size',data=newdf)
ax.set_xticklabels(ax.get_xticklabels(),rotation=40,ha='right')
plt.show()


In [ ]:
#Plotting histogram between body type and frequency
plt.figure(figsize=(12,6))
sns.histplot(newdf[newdf['fit'] == -1]['body_type'], color='white')
sns.histplot(newdf[newdf['fit'] == 0]['body_type'],color='red')
sns.histplot(newdf[newdf['fit'] == 1]['body_type'],color='green')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def draw_boxplots(cols, data, per_line=4):
    n = len(cols)
    per_line = 4
    for i in range(0, n, per_line):
        n_plots = per_line if n - i >= per_line else n % per_line
        fig, axes = plt.subplots(1, n_plots)
        plt.subplots_adjust(wspace=0.2)
        fig.set_figwidth(15)
        fig.set_figheight(4)
        for j in range(n_plots):
            sns.boxplot(data[cols[i + j]], ax=axes[j])
            axes[j].set_title(cols[i + j])  # set the title for the subplot
        plt.show()
draw_boxplots(numeric_features, newdf)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler

# Re-identifying numerical features to include only those suitable for the model
numeric_features_for_model = ['usr_weight_kg', 'usr_height_inchs', 'product_size', 'usr_age', 'bust_size_num', 'bust_size_cat', 'review_rating']

X = newdf[numeric_features_for_model]
y = newdf['fit']

# Handle any remaining NaN values in selected features by filling with the median
X = X.fillna(X.median())

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Shape of X_train_scaled: {X_train_scaled.shape}")
print(f"Shape of X_test_scaled: {X_test_scaled.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

In [ ]:
# Train the Logistic Regression model
log_reg_model = LogisticRegression(max_iter=5000, random_state=42)
log_reg_model.fit(X_train_scaled, y_train)

y_pred = log_reg_model.predict(X_test_scaled)

y_pred_proba = log_reg_model.predict_proba(X_test_scaled)

print("Model training complete. Now evaluating performance...")

In [ ]:
# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# Display the first few prediction probabilities
print("\nFirst 5 prediction probabilities (confidence scores for each class):\n", y_pred_proba[:5])

In [ ]:
#developing new feature BMI from height and weight.
import numpy as np
newdf['BMI'] = newdf['usr_weight_kg'] / np.power(newdf['usr_height_inchs'], 2)
newdf.drop(['usr_weight_kg', 'usr_height_inchs'], axis=1, inplace=True)

In [ ]:
#Adding BMI into dataframe and removing height and weight.
numeric_features.append('BMI')
numeric_features.remove('usr_weight_kg')
numeric_features.remove('usr_height_inchs')

In [ ]:
#One hot encoding for discrete features
body_type_col = newdf['body_type']
body_type_col_encoded = pd.get_dummies(body_type_col)
body_type_col_encoded.head()

In [ ]:
counts = newdf['product_category'].value_counts()

In [ ]:
newdf['product_category'].nunique()
threshold = 1000


repl = counts[counts <= threshold].index


top_n = 8
#One hot encoding for discrete features
prod_cats = newdf['product_category'].value_counts().index[:top_n].values
prod_cat_col = newdf['product_category'].apply(lambda x: x if x in prod_cats else 'other')
prod_cat_col_encoded = pd.get_dummies(newdf['product_category'].replace(repl, 'uncommon'))
prod_cat_col_encoded.sample(5)

# Handle the single null value in 'rented_for' by filling it with the mode
newdf['rented_for'] = newdf['rented_for'].fillna(newdf['rented_for'].mode()[0])

# Get value counts for 'rented_for'
rented_for_counts = newdf['rented_for'].value_counts()

# Determine threshold for 'uncommon' categories, if needed (e.g., less than 1% of total)
rented_for_threshold = len(newdf) * 0.01 # Example: categories less than 1% are uncommon
rented_for_repl = rented_for_counts[rented_for_counts <= rented_for_threshold].index

# One-hot encode 'rented_for' column, replacing uncommon categories with 'uncommon'
rented_for_col_encoded = pd.get_dummies(newdf['rented_for'].replace(rented_for_repl, 'uncommon'))


In [ ]:
#making a dataframe consisting of encoded discrete features.
dummy_features_df = pd.concat((prod_cat_col_encoded, rented_for_col_encoded, body_type_col_encoded), axis=1)
dummy_features_df.head()

In [ ]:
from sklearn.feature_selection import VarianceThreshold

# removing all features having in 80% samples either ones or zeros
sel = VarianceThreshold(threshold=(0.8 * (1 - 0.8)))
print(dummy_features_df.shape)
dummy_features = sel.fit_transform(dummy_features_df)
print(dummy_features.shape)

In [ ]:
#combining numeric features and discrete features.
import os

data = pd.concat((newdf[numeric_features], dummy_features_df), axis=1)

head, tail = os.path.split(json_file_path)
cleaned_data_fp = 'cleaned_' + tail.replace('json', 'csv')

data.to_csv(cleaned_data_fp, index=False)
data.head(3)

In [ ]:
#dropping two columns from dataframe.
from sklearn.model_selection import train_test_split
X = np.hstack((newdf[numeric_features].drop([target_feature,'bust_size_num', 'bust_size_cat'], axis=1), dummy_features))
y = newdf["fit"].astype('float32').values
X.shape, y.shape

((158219, 9), (158219,))

#oversampling the data using SMOTENC
from imblearn.over_sampling import SMOTENC
smote_nc = SMOTENC(categorical_features=[0, 2], random_state=0)
X_resampled, y_resampled = smote_nc.fit_resample(X, y)


from collections import Counter
print("Resampled dataset shape:", Counter(y_resampled))


In [ ]:
# split X and y into training and testing sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

In [ ]:
#using k-NN for calculating train and test scores by varying k.
from sklearn.neighbors import KNeighborsClassifier
test_scores_pre = []
train_scores_pre = []
for i in range(2,20):
    knn = KNeighborsClassifier(i)
    knn.fit(X_train,y_train)
    train_scores_pre.append(knn.score(X_train,y_train))
    test_scores_pre.append(knn.score(X_test,y_test))
max_test_score = max(test_scores_pre)
test_scores_ind = [i for i, v in enumerate(test_scores_pre) if v == max_test_score]
print('Max test score before hyperparamter tuning {:.2f} % and k = {}'.format(max_test_score*100,list(map(lambda x: x+1, test_scores_ind))))


In [ ]:
#plotting a graph for train and test scores for different values of k and finding which value of K will be the best.
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(20,7))
plt.title('')
sns.set(rc={'axes.facecolor':'lightblue', 'figure.facecolor':'white'})
sns.despine()
p = sns.lineplot(x=range(2,20),y=train_scores_pre,marker='*',label='Train Score',color='#03080c')
p = sns.lineplot(x=range(2,20),y=test_scores_pre,marker='o',label='Test Score', color= '#255075')

plt.xlabel('value of n_neighbors')
plt.ylabel('Prediction Accuracy')


In [ ]:
#As k=2 has highest test score we run the model for k=2.
knn=KNeighborsClassifier(2)
knn.fit(X_train,y_train)
y_pred = knn.predict(X_test)
from sklearn import metrics
cnf_matrix = metrics.confusion_matrix(y_test, y_pred)
p = sns.heatmap(pd.DataFrame(cnf_matrix), annot=True, cmap="YlGnBu" ,fmt='g')
plt.title('Confusion matrix', y=1.1)
plt.ylabel('Actual label')
plt.xlabel('Predicted label')

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))


In [ ]:
from sklearn.ensemble import RandomForestClassifier
rfc = RandomForestClassifier(n_estimators=50, random_state=2)
from sklearn.metrics import accuracy_score

rfc.fit(X_train, y_train)

# Make predictions on the test set
y_pred = rfc.predict(X_test)

# Calculate the accuracy of the model
accuracy = accuracy_score(y_test, y_pred)


In [ ]:
y_pred = rfc.predict(X_test)
from sklearn import metrics
cnf_matrix = metrics.confusion_matrix(y_test, y_pred)
p = sns.heatmap(pd.DataFrame(cnf_matrix), annot=True, cmap="YlGnBu" ,fmt='g')
plt.title('Confusion matrix', y=1.1)
plt.ylabel('Actual label')
plt.xlabel('Predicted label')


In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import numpy as np

# Generate some random data
X, y = make_classification(n_samples=1000, n_features=20, n_informative=10, n_classes=2, random_state=42)

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the parameter grid for hyperparameter tuning
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_features': ['sqrt', 'log2'],
    'max_depth': [5, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

# Create a random forest classifier
#rfc = RandomForestClassifier()

# Create a grid search object
grid_search = GridSearchCV(estimator=rfc, param_grid=param_grid, cv=5, n_jobs=-1)

# Fit the grid search object to the data
grid_search.fit(X_train, y_train)

# Print the best hyperparameters and their corresponding accuracy score
print("Best hyperparameters: ", grid_search.best_params_)


grid_results = grid_search.fit(X_train, y_train)
final_model = rfc.set_params(**grid_results.best_params_)
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)

print(classification_report(y_test,y_pred))
print(grid_results.best_params_)
